In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [4]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [5]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "vgg16",
    "learning_rate": 0.001,
    "epochs": 10,
    "pretrained":True
}

In [6]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

train_dir = "../flowers/train"
val_dir = "../flowers/val"

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [7]:
vgg16 = models.vgg16(pretrained=True)

for param in vgg16.parameters():
    param.requires_grad = False

vgg16.classifier = nn.Sequential(
    nn.Linear(7*7*512, 4096),
    nn.ReLU(inplace=True),
    nn.Linear(4096, 4096),
    nn.ReLU(inplace=True),
    nn.Linear(4096, 5)
)

for param in vgg16.classifier[-1].parameters():
    param.requires_grad = True

gpu = torch.device("cuda")
vgg16 = vgg16.to(gpu)


/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.parameters(), lr=model_config["learning_rate"])
epochs = model_config["epochs"]

In [9]:
def train_model(model, train_dataloader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}")

In [10]:
train_model(vgg16, train_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.9257216778735953
Epoch: 2, Training Loss: 0.2695045632261976
Epoch: 3, Training Loss: 0.18253871520133122
Epoch: 4, Training Loss: 0.14921155101513928
Epoch: 5, Training Loss: 0.16488180052920323
Epoch: 6, Training Loss: 0.11818932872615026
Epoch: 7, Training Loss: 0.04783728309040048
Epoch: 8, Training Loss: 0.10010273075374154
Epoch: 9, Training Loss: 0.061305129830797465
Epoch: 10, Training Loss: 0.07597526960050521


In [11]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [12]:
accuracy = validate_model(vgg16, val_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 91.29574678536103
